In [4]:
import numpy as np

def run_gmm_em(data, steps=3):
    # Convert input to a standard numpy array for speed
    x = np.asarray(data)
    n = len(x)

    # Iteration 0: Establish base parameters / initial guesses
    mu1 = 62.0  # Base guess for child group
    mu2 = 75.0  # Base guess for pro basketball group
    var1 = 9.0
    var2 = 9.0
    pi1 = 0.5
    pi2 = 0.5

    print("--- Iteration 0 (Initialization) ---")
    print(f"mu1: {mu1:.2f}, mu2: {mu2:.2f}, var1: {var1:.2f}, var2: {var2:.2f}, pi1: {pi1:.2f}, pi2: {pi2:.2f}\n")

    for epoch in range(1, steps + 1):
        v1 = max(var1, 1e-6)
        v2 = max(var2, 1e-6)

        pdf1 = (1.0 / np.sqrt(2 * np.pi * v1)) * np.exp(-((x - mu1) ** 2) / (2 * v1))
        pdf2 = (1.0 / np.sqrt(2 * np.pi * v2)) * np.exp(-((x - mu2) ** 2) / (2 * v2))

        # EXPECTATION (E-STEP): Calculate weights/responsibilities
        mixed_p1 = pi1 * pdf1
        mixed_p2 = pi2 * pdf2
        total_density = mixed_p1 + mixed_p2

        gamma1 = mixed_p1 / total_density
        gamma2 = mixed_p2 / total_density

        # Calculate overall system performance metric (Log-Likelihood)
        ll = np.sum(np.log(total_density))

        weight_sum1 = np.sum(gamma1)
        weight_sum2 = np.sum(gamma2)

        mu1 = np.sum(gamma1 * x) / weight_sum1
        mu2 = np.sum(gamma2 * x) / weight_sum2

        var1 = np.sum(gamma1 * ((x - mu1) ** 2)) / weight_sum1
        var2 = np.sum(gamma2 * ((x - mu2) ** 2)) / weight_sum2

        pi1 = weight_sum1 / n
        pi2 = weight_sum2 / n

        # Format exact outputs requested for presentation tracking tables
        print(f"--- Iteration {epoch} ---")
        print(f"μ1 (Children): {mu1:.4f} | μ2 (Pros): {mu2:.4f}")
        print(f"σ1²: {var1:.4f} | σ2²: {var2:.4f}")
        print(f"π1: {pi1:.4f} | π2: {pi2:.4f}")
        print(f"Log-Likelihood: {ll:.4f}\n")

    return mu1, mu2, var1, var2, pi1, pi2

if __name__ == "__main__":
    np.random.seed(42)
    sample_heights = np.concatenate([
        np.random.normal(64, 2.5, 100),  # Underlyng children cluster
        np.random.normal(76, 3.0, 50)    # Underlying pros cluster
    ])

    final_m1, final_m2, final_v1, final_v2, final_p1, final_p2 = run_gmm_em(sample_heights, steps=2)

    # Target Evaluation Hook for Live Presentation Query
    test_val = 72.0

    p1_raw = (1.0 / np.sqrt(2 * np.pi * final_v1)) * np.exp(-((test_val - final_m1) ** 2) / (2 * final_v1)) * final_p1
    p2_raw = (1.0 / np.sqrt(2 * np.pi * final_v2)) * np.exp(-((test_val - final_m2) ** 2) / (2 * final_v2)) * final_p2
    scale = p1_raw + p2_raw

    print("==============================================")
    print(f"LIVE TEST CASE PREDICTION FOR HEIGHT: {test_val} inches")
    print(f"Probability it is a Child: {p1_raw / scale:.4f}")
    print(f"Probability it is a Basketball Player: {p2_raw / scale:.4f}")
    print("==============================================")

--- Iteration 0 (Initialization) ---
mu1: 62.00, mu2: 75.00, var1: 9.00, var2: 9.00, pi1: 0.50, pi2: 0.50

--- Iteration 1 ---
μ1 (Children): 63.6351 | μ2 (Pros): 75.4267
σ1²: 4.8526 | σ2²: 12.2921
π1: 0.6478 | π2: 0.3522
Log-Likelihood: -475.5234

--- Iteration 2 ---
μ1 (Children): 63.6770 | μ2 (Pros): 75.5878
σ1²: 4.9600 | σ2²: 11.2472
π1: 0.6549 | π2: 0.3451
Log-Likelihood: -444.4825

LIVE TEST CASE PREDICTION FOR HEIGHT: 72.0 inches
Probability it is a Child: 0.0047
Probability it is a Basketball Player: 0.9953
